# Exercise 01 — your first Kafka producer

**Goal:** send a single event into Redpanda, verify it arrived in three ways (delivery callback, Redpanda Console, partition+offset numbers), and explore what happens when things go wrong.

**Use case:** a food-delivery app. Riders send a GPS ping every few seconds to the topic `rider-locations`; customers place orders, which go to the topic `orders`. Each ping is keyed by rider id (`rider_7`, `rider_3`, …) so all pings of the same rider land in the same partition, in order.

**Stuck?** Compare with [`../02_demo/demo_produce.ipynb`](../02_demo/demo_produce.ipynb). Full reference solution in [`../04_solution/exercise_01_produce_single.ipynb`](../04_solution/exercise_01_produce_single.ipynb).

## Step 1 — connect to the broker

**Concept check.** A *broker* is the server program that stores Kafka messages. Producers and consumers connect to it over TCP. Even if a real cluster has many brokers, you only need to give the producer *one* address as `bootstrap.servers` — the broker then teaches the producer about the rest of the cluster automatically.

Inside our docker network the broker is at `redpanda:29092`. The Codespace's port-forward `localhost:9092` would *also* work from your host, but **not** from another container — that's why all our notebooks use the internal address.

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'   # internal listener inside the docker network

conf = {
    'bootstrap.servers': BROKER,
    'client.id':         'python-producer',  # shows up in broker logs
}
producer = Producer(conf)
print(f'Producer connected to {BROKER}')

### Check the topics

Kafka splits each topic into *partitions*. Redpanda would create a topic automatically on the first write, but only with **one** partition, and then every key would land on partition 0. The cell below makes sure both topics exist with **3 partitions**. If you ran the smoke test, it only confirms that. Running it again is harmless.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic, NewPartitions

PARTITIONS = 3
admin    = AdminClient({'bootstrap.servers': BROKER})
existing = admin.list_topics(timeout=5).topics

for name in ['rider-locations', 'orders']:
    if name not in existing:
        admin.create_topics([NewTopic(name, num_partitions=PARTITIONS)])[name].result()
        print(f'OK — created {name!r} with {PARTITIONS} partitions')
    elif len(existing[name].partitions) < PARTITIONS:
        admin.create_partitions([NewPartitions(name, PARTITIONS)])[name].result()
        print(f'OK — {name!r} grown to {PARTITIONS} partitions')
    else:
        print(f'OK — {name!r} already has {len(existing[name].partitions)} partitions')

## Step 2 — build the event

Every Kafka message has three pieces:

| Piece | Purpose | Example |
|---|---|---|
| **Topic** | the *category* — like a folder | `rider-locations` |
| **Key**   | routes the event to a partition | `rider_7` |
| **Value** | the payload (any bytes) | `'{"lat": 47.0502, ...}'` |

**Why does the key matter?** Kafka splits each topic into *partitions* for parallelism. The producer hashes the key and uses `hash(key) % num_partitions` to decide which one. Same key → same partition → messages with that key arrive at the consumer **in order**. No key → spread over the partitions → no ordering guarantee. For the live map that matters: if rider 7's pings arrive out of order, the icon jumps back and forth.

**Task — fill in the two missing values below.**

In [ ]:
topic_name    = 'rider-locations'
message_key   = 'rider_7'
message_value = json.dumps({
    'rider_id':  'rider_7',
    'lat':       None,    # TODO: a latitude in Lucerne, e.g. 47.0502 (main station)
    'lon':       None,    # TODO: the matching longitude, e.g. 8.3093
    'speed_kmh': 18.4,
    'timestamp': time.time(),
})

print(f'Topic:  {topic_name}')
print(f'Key:    {message_key}')
print(f'Value:  {message_value}')

## Step 3 — send the event

`producer.produce(...)` doesn't actually send anything yet — it puts the message into an in-memory queue and returns immediately. The real send happens in a background thread. That's good for throughput (thousands of messages/second) but means we have to be careful at shutdown.

Two follow-ups:

- **`callback`** — Kafka calls this function once the broker has confirmed (or rejected) each message. Useful for logging failures.
- **`flush()`** — blocks until the queue is empty. *Always* call this before the program exits, otherwise messages can be silently lost.

**Task — call `producer.produce(...)` and `producer.flush()`.** Both key and value must be `bytes`, so call `.encode('utf-8')` on the strings.

In [ ]:
def delivery_report(err, msg):
    """Called by Kafka once delivery is confirmed (or has failed)."""
    if err:
        print(f'Delivery failed: {err}')
    else:
        print(f'Delivered to {msg.topic()} '
              f'[partition {msg.partition()}] offset {msg.offset()}')

# TODO: call producer.produce() with topic_name, message_key,
#       message_value (both .encode()'d) and callback=delivery_report


# TODO: call producer.flush() — why is this needed?
print('Done.')

**Verify it worked** in three places:

1. The `Delivered to rider-locations [partition X] offset Y` line above.
2. **Redpanda Console** (port `8080` in *PORTS*) → *Topics* → `rider-locations` → *Messages*. Your event is at the bottom.
3. From a terminal: `rpk topic consume rider-locations -X brokers=redpanda:29092 --num 1`.

## Task A — second key, observe the partition change

Send another ping to `rider-locations`, but for rider `rider_3` (key `rider_3`) this time. Compare the *partition* numbers in the output:

- Different partition? That's the normal case — keys are spread across partitions.
- Now try `rider_5` too. Same partition as `rider_7`? That's a hash collision, and it is perfectly normal: with 3 partitions and hundreds of riders, every partition holds many keys. What Kafka guarantees is that each individual key always sticks to one partition.

Re-run the cell several times: the partition for `rider_3` *never* changes. Re-create the topic with a different number of partitions and the mapping changes — that's why partition counts are usually fixed at topic creation time.

In [ ]:
# TODO: build message_value_3 for rider_3 (similar to above)
# TODO: producer.produce(...) with key='rider_3', then flush

## Task B — write to a different topic

Anna orders a pizza. Send an event to topic `orders` with key `order-4711` and this value:

```json
{"order_id": "order-4711", "customer": "Anna", "restaurant": "Pizzeria Roma",
 "total_chf": 32.50, "status": "placed", "timestamp": ...}
```

This shows that **topics are independent**: each topic has its own partitions and its own offsets (Anna's order starts at offset 0 of `orders`, no matter how many pings `rider-locations` already holds), and consumers subscribe per topic. It also shows that each topic picks the key that keeps *its* related events together: all pings of one rider, all status changes of one order.

In [ ]:
# TODO: send Anna's order to topic 'orders' with key 'order-4711'

## Task C — what happens when the broker is unreachable?

Run the cell below. It points the producer at a non-existent address, so delivery will *fail*.

**Watch for two things:**

1. The error message — it tells you exactly why delivery failed.
2. The duration — `flush(timeout=5)` waits up to 5 s. Without the `message.timeout.ms` setting, librdkafka would retry for **5 minutes** by default.

**Think about it:** in a production system, what would you do with messages that fail to deliver? Drop them? Retry forever? Persist them to a separate "dead-letter" topic for inspection? — there is no single right answer; it depends on whether the data is recoverable and whether duplicates are acceptable. A lost GPS ping is replaced by the next one a few seconds later; a lost order is a hungry customer.

In [ ]:
errors = []
def error_cb(err, msg):
    if err:
        errors.append(str(err))

p_broken = Producer({
    'bootstrap.servers':  'localhost:9999',   # nothing listens here
    'message.timeout.ms': 4000,               # give up after 4 s
    'socket.timeout.ms':  2000,
})
p_broken.produce('rider-locations', key=b'rider_7', value=b'{"test": true}',
                 callback=error_cb)
p_broken.flush(timeout=5)

print(errors[0] if errors else 'Delivered (unexpected)')

## What you learned

- The three pieces of a Kafka message: **topic, key, value**.
- Why the key matters: it determines the partition, and partitions preserve ordering.
- Why we always call `producer.flush()` before exiting.
- That delivery is *asynchronous* and we use a callback to learn about success or failure.
- That broker failures don't crash the producer; they show up via the callback.

Next: read what you produced. Continue with [`exercise_02_consume_basic.ipynb`](exercise_02_consume_basic.ipynb).